Ingest Raw Data (Bronze Layer)

In [0]:
import requests
import json
from pyspark.sql import functions as F
from datetime import datetime, timezone
import time

BASE_URL = "https://fantasy.premierleague.com/api"
INGESTION_TS = datetime.now(timezone.utc).isoformat()

def fetch_api(endpoint):
  """Fetch JSON form FPL API and return as a Python dict."""
  response = requests.get(f"{BASE_URL}/{endpoint}")
  response.raise_for_status()
  return response.json()

Ingest Bootstrap Static

In [0]:
import json

bootstrap = fetch_api("bootstrap-static/")

datasets = {
    "players": bootstrap["elements"],
    "teams": bootstrap["teams"],
    "events": bootstrap["events"],
    "positions": bootstrap["element_types"],
}

for name, data in datasets.items():
    path = f"/Volumes/fpl_project/bronze/raw_files/{name}.json"

    with open(path, "w") as f:
        for record in data:
            f.write(json.dumps(record) + "\n")

    df = spark.read.json(path) \
        .withColumn("_ingestion_ts", F.lit(INGESTION_TS)) \
        .withColumn("_source", F.lit("bootstrap-static"))

    df.write.mode("append").format("delta") \
        .saveAsTable(f"fpl_project.bronze.{name}")

print("Bootstrap ingestion complete.")

Fixtures

In [0]:
fixtures = fetch_api("fixtures/")

path = "/Volumes/fpl_project/bronze/raw_files/fixtures.json"
with open(path, "w") as f:
    for record in fixtures:
        f.write(json.dumps(record) + "\n")

fixtures_df = spark.read.json(path) \
    .withColumn("_ingestion_ts", F.lit(INGESTION_TS)) \
    .withColumn("_source", F.lit("fixtures"))

fixtures_df.write.mode("append").format("delta") \
    .saveAsTable("fpl_project.bronze.fixtures")

print("Fixtures ingestion complete.")

Player Histories

In [0]:
player_ids = [
    row.id for row in
    spark.table("fpl_project.bronze.players")
    .select("id").distinct().collect()
]

all_histories = []
for pid in player_ids:
    try:
        summary = fetch_api(f"element-summary/{pid}/")
        for h in summary.get("history", []):
            h["player_id"] = pid
            all_histories.append(h)
        time.sleep(0.5)
    except Exception as e:
        print(f"Failed for player {pid}: {e}")

if all_histories:
    path = "/Volumes/fpl_project/bronze/raw_files/player_history.json"
    with open(path, "w") as f:
        for record in all_histories:
            f.write(json.dumps(record) + "\n")

    history_df = spark.read.json(path) \
        .withColumn("_ingestion_ts", F.lit(INGESTION_TS))
    history_df.write.mode("append").format("delta") \
        .saveAsTable("fpl_project.bronze.player_history")

print(f"Ingested history for {len(all_histories)} player-gameweek records.")